I tired of huge 30B models, so llama will be 8B without quantizing.

In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "3" #without it gptq will try to start on all gpus
from transformers import AutoModelForCausalLM
import torch
from dotenv import load_dotenv
load_dotenv("/glazkov-dev/.env") #for hf token and cache directory
# from transformers import BitsAndBytesConfig
# from gptqmodel import GPTQModel

ImportError: libcusparseLt.so.0: cannot open shared object file: No such file or directory

In [ ]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct" 
# quantization_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_compute_dtype=torch.float16,  # Compute in float16 for speed
#         bnb_4bit_use_double_quant=True,  # Double quantization for extra memory savings
#         # Normalized float 4-bit (optimal for LLMs)
#         bnb_4bit_quant_type="nf4",
#         llm_int8_skip_modules=[
#             "lm_head",
#             "mlp.gate", #because DeepSeekV2ForCausalLM has a f.linear(self.gate.weight...) instead of self.gate()
#             #it couldn't be substituded with Linear4bit
#         ],
#     )

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    #quantization_config=quantization_config,
    #dtype=torch.bfloat16,
    device_map="cuda:3",
    # cache_dir="/glazkov-dev/.cache",
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (rotary_fn): Func()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): Llama

In [ ]:
import transformer_lens
print(transformer_lens.__file__)

/glazkov-dev/TransformerLens/transformer_lens/__init__.py


In [ ]:
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers(
    MODEL,
    hf_model=model,
    dtype=torch.float16,
)

In [ ]:
from utils.mmlu_batch_generator import MMLUBatchGenerator
SELECTED_TASKS = ["anatomy",
            # "conceptual_physics",
            # "human_sexuality",
            "machine_learning",
            "management",
            # "marketing",
            # "nutrition",
             "philosophy",
            # "us_foreign_policy",
              "world_religions"]
gen = MMLUBatchGenerator(subjects=SELECTED_TASKS, split='validation', batch_size=16, include_metadata=False)

Initialized MMLU batch generator with 5 subjects
Split: validation, Batch size: 16


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL,
                                                device_map='cuda:3',
                                                )
tokenizer.pad_token = tokenizer.eos_token

for batch in gen:
    inputs = tokenizer(batch, return_tensors="pt",
            padding=True,  # Pad to longest in batch
            truncation=True,
            max_length=512,  # Adjust based on your needs
            add_special_tokens=True
    )  
    inputs = {k: v.to(bridge.device) for k, v in inputs.items()}
    print(inputs.keys())
    bridge_outputs, cache = bridge.run_with_cache(inputs['input_ids'], attention_mask=inputs['attention_mask'])
    print(bridge_outputs)  # Example: (batch_size, sequence_length, vocab_size)
    break  # Remove this break to process all batches

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.



📚 Loading subject 1/5: anatomy
  ✓ Created batch with 14 questions
dict_keys(['input_ids', 'attention_mask'])
tensor([[[-3.0781,  1.5391,  7.3750,  ...,  1.7344,  1.7344,  1.7344],
         [ 0.0562, -0.5312, -0.8359,  ..., -7.3125, -7.3125, -7.3125],
         [ 3.0156,  2.0000,  0.5273,  ..., -1.1719, -1.1719, -1.1719],
         ...,
         [ 5.5000,  3.1094,  4.6562,  ..., -3.6719, -3.6719, -3.6719],
         [ 5.5625,  3.3125,  4.7188,  ..., -3.8594, -3.8594, -3.8594],
         [ 6.0312,  3.4531,  5.0312,  ..., -4.0312, -4.0312, -4.0312]],

        [[-3.0781,  1.5391,  7.3750,  ...,  1.7344,  1.7344,  1.7344],
         [ 0.0562, -0.5312, -0.8359,  ..., -7.3125, -7.3125, -7.3125],
         [ 3.0156,  2.0000,  0.5273,  ..., -1.1719, -1.1719, -1.1719],
         ...,
         [ 5.9375,  3.3125,  2.5000,  ..., -5.3750, -5.3750, -5.3750],
         [ 6.0625,  3.3906,  2.3438,  ..., -5.4688, -5.4688, -5.4688],
         [ 5.9688,  3.6406,  2.5312,  ..., -5.4062, -5.4062, -5.4062]],

     

In [ ]:
bridge_outputs.shape

torch.Size([14, 34, 128256])